# 📋 자산 입력 UI
보유 자산을 화면에서 **추가 / 수정 / 삭제**하고 **assets.csv**에 저장합니다.
- **매수일**: 모든 자산 공통 (실제 매수한 날짜)
- **만기일**: 예금·채권·펀드만 입력 — 만기일이 오늘 이전이면 자동으로 비활성 처리
- 비활성 자산은 모든 계산(버킷/리밸런싱/리스크)에서 자동 제외됩니다.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pathlib import Path
from datetime import date

CSV_PATH        = Path('assets.csv')
ASSET_TYPES     = ['cash', 'bond', 'fund', 'equity', 'income']
EXPIRABLE_TYPES = ['cash', 'bond', 'fund']   # 만기일 입력 대상
TODAY           = date.today()

COLUMNS = [
    'account_name','asset_name','ticker','asset_type',
    'quantity','unit_price','current_value',
    'purchase_date','maturity_date','is_active'
]

# ── CSV 로드 ────────────────────────────────────────────────
def load_df():
    if CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH, dtype={'ticker': str})
        if 'is_active'     not in df.columns: df['is_active']     = 1
        if 'maturity_date' not in df.columns: df['maturity_date'] = ''
        df['is_active']     = df['is_active'].fillna(1).astype(int)
        df['maturity_date'] = df['maturity_date'].fillna('').astype(str).str.strip()
        return df.reset_index(drop=True)
    return pd.DataFrame(columns=COLUMNS)

df = load_df()

# ── 자동 만료 감지 (maturity_date 기준) ─────────────────────
def auto_expire(df):
    count = 0
    for i, row in df.iterrows():
        mat_str = str(row.get('maturity_date', '')).strip()
        if not mat_str or mat_str in ('nan', 'NaT', ''):
            continue
        if str(row['asset_type']) in EXPIRABLE_TYPES and int(row.get('is_active', 1)) == 1:
            try:
                if date.fromisoformat(mat_str) < TODAY:
                    df.at[i, 'is_active'] = 0
                    count += 1
            except:
                pass
    return df, count

df, expired_count = auto_expire(df)

display(HTML('<h2 style="color:#2c3e50;margin-bottom:4px">📋 자산 입력 UI</h2><hr style="margin-top:0">'))
if expired_count > 0:
    display(HTML(
        f'<div style="background:#fef9e7;border:1px solid #f39c12;border-radius:6px;'
        f'padding:10px 14px;margin-bottom:8px">'
        f'⚠️ <b>{expired_count}개 자산</b>의 만기일이 지나 자동으로 비활성 처리됐습니다. '
        f'「💾 CSV 저장」을 눌러 반영하세요.</div>'
    ))

In [2]:
# ── 자산 목록 테이블 ─────────────────────────────────────────
table_out = widgets.Output()
msg_out   = widgets.Output()

def fmt_num(v):
    try:    return f'{int(float(v)):,}'
    except: return str(v)

def build_table_html(rows_df, title, header_color):
    if rows_df.empty:
        return f'<p style="color:#7f8c8d;font-size:13px">{title}: 없음</p>'

    rows_html = ''
    for real_idx, row in rows_df.iterrows():
        bg = '#f9f9f9' if real_idx % 2 == 0 else '#ffffff'

        # 만기일 표시 및 임박 경고
        mat_str  = str(row.get('maturity_date', '')).strip()
        mat_disp = ''
        if mat_str and mat_str not in ('nan', 'NaT', ''):
            mat_disp = mat_str
            try:
                days_left = (date.fromisoformat(mat_str) - TODAY).days
                if 0 <= days_left <= 30:
                    mat_disp += f' <span style="color:#e67e22;font-size:11px">⏰D-{days_left}</span>'
                elif days_left < 0:
                    mat_disp += ' <span style="color:#e74c3c;font-size:11px">만료</span>'
            except:
                pass

        type_color = {'cash':'#2980b9','bond':'#8e44ad','fund':'#16a085',
                      'equity':'#e67e22','income':'#27ae60'}.get(str(row['asset_type']),'#555')
        rows_html += (
            f'<tr style="background:{bg}">'
            f'<td style="padding:5px 8px;color:#888">{real_idx+1}</td>'
            f'<td style="padding:5px 8px">{row["account_name"]}</td>'
            f'<td style="padding:5px 8px;font-weight:bold">{row["asset_name"]}</td>'
            f'<td style="padding:5px 8px;color:#2980b9">{row["ticker"] if str(row["ticker"]) != "nan" else ""}</td>'
            f'<td style="padding:5px 8px"><span style="background:{type_color}22;color:{type_color};'
            f'padding:2px 8px;border-radius:10px;font-size:11px;font-weight:bold">{row["asset_type"]}</span></td>'
            f'<td style="padding:5px 8px;text-align:right">{fmt_num(row["quantity"])}</td>'
            f'<td style="padding:5px 8px;text-align:right">{fmt_num(row["unit_price"])}</td>'
            f'<td style="padding:5px 8px;text-align:right;font-weight:bold;color:#27ae60">{fmt_num(row["current_value"])}</td>'
            f'<td style="padding:5px 8px;color:#7f8c8d;font-size:12px">{row["purchase_date"]}</td>'
            f'<td style="padding:5px 8px;font-size:12px">{mat_disp}</td>'
            f'</tr>'
        )

    total = rows_df['current_value'].apply(
        lambda x: float(str(x).replace(',','')) if str(x).strip() else 0
    ).sum()

    return (
        f'<div style="margin-bottom:20px">'
        f'<div style="font-size:15px;font-weight:bold;color:{header_color};margin-bottom:6px">{title}</div>'
        '<div style="overflow-x:auto">'
        '<table style="border-collapse:collapse;width:100%;font-size:13px">'
        f'<thead><tr style="background:{header_color};color:white">'
        '<th style="padding:7px 8px">#</th>'
        '<th style="padding:7px 8px">계좌</th>'
        '<th style="padding:7px 8px">자산명</th>'
        '<th style="padding:7px 8px">티커</th>'
        '<th style="padding:7px 8px">유형</th>'
        '<th style="padding:7px 8px">수량</th>'
        '<th style="padding:7px 8px">단가(원)</th>'
        '<th style="padding:7px 8px">평가금액(원)</th>'
        '<th style="padding:7px 8px">매수일</th>'
        '<th style="padding:7px 8px">만기일</th>'
        '</tr></thead>'
        f'<tbody>{rows_html}</tbody>'
        f'<tfoot><tr style="background:#ecf0f1">'
        f'<td colspan="7" style="padding:7px 8px;text-align:right;font-weight:bold">소계</td>'
        f'<td style="padding:7px 8px;text-align:right;font-weight:bold;color:#27ae60;font-size:14px">{total:,.0f}원</td>'
        f'<td colspan="2"></td></tr></tfoot>'
        '</table></div></div>'
    )

def render_table():
    with table_out:
        clear_output(wait=True)
        if df.empty:
            display(HTML('<p style="color:#7f8c8d">아직 자산이 없습니다. 아래에서 추가해 주세요.</p>'))
            return

        active_df   = df[df['is_active'] == 1]
        inactive_df = df[df['is_active'] == 0]
        total_active = active_df['current_value'].apply(
            lambda x: float(str(x).replace(',','')) if str(x).strip() else 0
        ).sum()

        summary = (
            f'<div style="background:#eaf4fb;border-radius:8px;padding:10px 16px;'
            f'margin-bottom:14px;font-size:13px">'
            f'활성 자산: <b>{len(active_df)}개</b> / '
            f'총 평가금액: <b style="color:#27ae60">{total_active:,.0f}원</b>'
            f'&nbsp;│&nbsp; 비활성(만료/제외): <b style="color:#e74c3c">{len(inactive_df)}개</b>'
            f'</div>'
        )
        display(HTML(summary))
        display(HTML(build_table_html(active_df,   '✅ 활성 자산', '#2c3e50')))
        if not inactive_df.empty:
            display(HTML(build_table_html(inactive_df, '🚫 비활성 자산 (만료·제외)', '#95a5a6')))

render_table()
display(table_out)

#,계좌,자산명,티커,유형,수량,단가(원),평가금액(원),매수일,만기일
1,연금저축,서울도시철도 24-06,,bond,0,0,"3,850,000",2024-06-12,2031-06-30
2,연금저축,키움글로벌얼터너티브증권(혼합-재)-C-P2,,fund,0,0,"433,400",2024-06-12,2040-12-30
3,연금저축,NH 저축은행 정기예금DC,,cash,0,0,"40,938,608",2025-05-28,2026-05-28 ⏰D-7
4,연금저축,KODEX 미국배당다우존스,489250.0,equity,"1,882","13,230","24,898,860",2025-06-05,
5,연금저축,KODEX 200,69500.0,equity,961,"113,340","108,919,740",2025-07-20,
6,연금저축,삼성증권 디폴트옵션 안정형 포트폴리오,,cash,0,0,"1,341,221",2025-07-20,2030-07-20
7,연금저축,SBI저축은행 퇴직연금 IRP,,cash,0,0,"27,921,481",2025-10-14,2026-10-14
8,연금저축,삼성화재 이율보증형(3년),,cash,0,0,"22,605,072",2025-11-21,2028-11-21
9,연금저축,푸른저축은행 퇴직연금정기예금 IRP (1년),,cash,0,0,"39,487,027",2025-12-10,2026-12-10
10,연금저축,KODEX 미국S&P500,379800.0,equity,895,"25,080","22,446,600",2026-12-31,


Output()

In [4]:
# ── 입력 폼 ─────────────────────────────────────────────────
display(HTML('<h3 style="color:#34495e;margin-top:20px">✏️ 자산 추가 / 수정</h3>'))
display(HTML(
    '<p style="color:#7f8c8d;font-size:12px">'
    '행 번호 입력 시 기존 값 자동 채우기. 0이면 신규 추가.<br>'
    '만기일은 <b>예금·채권·펀드</b>만 입력하세요 (주식·배당은 불필요).'
    '</p>'
))

s  = {'description_width': '150px'}
lw = widgets.Layout(width='420px')
lw_sm = widgets.Layout(width='280px')

w_row_no     = widgets.IntText(value=0,             description='행 번호 (0=신규)',   style=s, layout=lw_sm)
w_account    = widgets.Text(value='',               description='계좌명',             placeholder='예: IRP계좌, 일반계좌', style=s, layout=lw)
w_asset_name = widgets.Text(value='',               description='자산명',             placeholder='예: KODEX 200', style=s, layout=lw)
w_ticker     = widgets.Text(value='',               description='티커',               placeholder='예: 069500 (없으면 빈칸)', style=s, layout=lw)
w_type       = widgets.Dropdown(options=ASSET_TYPES, value='equity', description='자산 유형', style=s, layout=lw)
w_qty        = widgets.FloatText(value=0,           description='수량',               style=s, layout=lw)
w_unit       = widgets.IntText(value=0,             description='단가 (원)',           style=s, layout=lw)
w_cur_val    = widgets.IntText(value=0,             description='평가금액 (원)',        style=s, layout=lw)
w_buy_date   = widgets.Text(value=str(TODAY),       description='매수일',              placeholder='YYYY-MM-DD', style=s, layout=lw)
w_mat_date   = widgets.Text(value='',               description='만기일 (예금·채권·펀드)', placeholder='YYYY-MM-DD  (해당 없으면 빈칸)', style=s, layout=lw)

# 만기일 필드: expirable 타입일 때만 보임
mat_box = widgets.VBox([w_mat_date])

def toggle_maturity(change):
    is_exp = change['new'] in EXPIRABLE_TYPES
    mat_box.layout.display = '' if is_exp else 'none'
    if not is_exp:
        w_mat_date.value = ''

w_type.observe(toggle_maturity, names='value')
# 초기 상태
mat_box.layout.display = 'none'   # 기본값 equity → 숨김

# 수량×단가 → 평가금액 자동 계산
def auto_calc(*args):
    try:
        if w_qty.value > 0 and w_unit.value > 0:
            w_cur_val.value = int(w_qty.value * w_unit.value)
    except:
        pass

w_qty.observe(auto_calc, names='value')
w_unit.observe(auto_calc, names='value')

# 행 번호 선택 시 기존 값 자동 채우기
def on_row_select(change):
    idx = change['new'] - 1
    if 0 <= idx < len(df):
        row = df.iloc[idx]
        w_account.value    = str(row['account_name'])
        w_asset_name.value = str(row['asset_name'])
        w_ticker.value     = str(row['ticker']) if str(row['ticker']) != 'nan' else ''
        t = str(row['asset_type'])
        if t in ASSET_TYPES: w_type.value = t
        try:    w_qty.value     = float(row['quantity'])
        except: w_qty.value     = 0
        try:    w_unit.value    = int(float(str(row['unit_price']).replace(',','')))
        except: w_unit.value    = 0
        try:    w_cur_val.value = int(float(str(row['current_value']).replace(',','')))
        except: w_cur_val.value = 0
        w_buy_date.value = str(row['purchase_date'])
        mat = str(row.get('maturity_date', '')).strip()
        w_mat_date.value = mat if mat not in ('nan', 'NaT', '') else ''

w_row_no.observe(on_row_select, names='value')

display(w_row_no)
display(widgets.HTML('<hr style="margin:8px 0">'))
display(w_account, w_asset_name, w_ticker, w_type,
        w_qty, w_unit, w_cur_val, w_buy_date, mat_box)

IntText(value=0, description='행 번호 (0=신규)', layout=Layout(width='280px'), style=DescriptionStyle(description_w…

HTML(value='<hr style="margin:8px 0">')

Text(value='', description='계좌명', layout=Layout(width='420px'), placeholder='예: IRP계좌, 일반계좌', style=TextStyle(…

Text(value='', description='자산명', layout=Layout(width='420px'), placeholder='예: KODEX 200', style=TextStyle(de…

Text(value='', description='티커', layout=Layout(width='420px'), placeholder='예: 069500 (없으면 빈칸)', style=TextSty…

Dropdown(description='자산 유형', index=3, layout=Layout(width='420px'), options=('cash', 'bond', 'fund', 'equity'…

FloatText(value=0.0, description='수량', layout=Layout(width='420px'), style=DescriptionStyle(description_width=…

IntText(value=0, description='단가 (원)', layout=Layout(width='420px'), style=DescriptionStyle(description_width=…

IntText(value=0, description='평가금액 (원)', layout=Layout(width='420px'), style=DescriptionStyle(description_widt…

Text(value='2026-05-21', description='매수일', layout=Layout(width='420px'), placeholder='YYYY-MM-DD', style=Text…

In [5]:
# ── 버튼 ───────────────────────────────────────────────────
btn_add   = widgets.Button(description='➕ 추가/수정',    button_style='success',
                            layout=widgets.Layout(width='140px', height='40px'))
btn_del   = widgets.Button(description='🗑 삭제',         button_style='danger',
                            layout=widgets.Layout(width='140px', height='40px'))
btn_deact = widgets.Button(description='🚫 비활성화',     button_style='warning',
                            layout=widgets.Layout(width='140px', height='40px'))
btn_react = widgets.Button(description='✅ 다시 활성화',  button_style='info',
                            layout=widgets.Layout(width='140px', height='40px'))
btn_save  = widgets.Button(description='💾 CSV 저장',     button_style='',
                            layout=widgets.Layout(width='140px', height='40px'))
btn_clr   = widgets.Button(description='🧹 폼 초기화',    button_style='',
                            layout=widgets.Layout(width='140px', height='40px'))

# 삭제 확인 위젯
btn_del_confirm = widgets.Button(description='⚠️ 정말 삭제',  button_style='danger',
                                  layout=widgets.Layout(width='140px', height='40px'))
btn_del_cancel  = widgets.Button(description='❌ 취소',        button_style='',
                                  layout=widgets.Layout(width='140px', height='40px'))
confirm_box = widgets.VBox([])

def clear_form():
    w_row_no.value     = 0
    w_account.value    = ''
    w_asset_name.value = ''
    w_ticker.value     = ''
    w_type.value       = 'equity'
    w_qty.value        = 0
    w_unit.value       = 0
    w_cur_val.value    = 0
    w_buy_date.value   = str(TODAY)
    w_mat_date.value   = ''

def make_row(is_active_val=1):
    return {
        'account_name':  w_account.value.strip(),
        'asset_name':    w_asset_name.value.strip(),
        'ticker':        w_ticker.value.strip(),
        'asset_type':    w_type.value,
        'quantity':      w_qty.value,
        'unit_price':    w_unit.value,
        'current_value': w_cur_val.value,
        'purchase_date': w_buy_date.value.strip(),
        'maturity_date': w_mat_date.value.strip(),
        'is_active':     is_active_val,
    }

def on_add(b):
    global df
    confirm_box.children = []  # 확인창 숨기기
    with msg_out:
        clear_output()
        if not w_asset_name.value.strip():
            display(HTML('<div style="color:#e74c3c">❌ 자산명을 입력해 주세요.</div>'))
            return
        idx = w_row_no.value - 1
        row = make_row()
        if 0 <= idx < len(df):
            row['is_active'] = int(df.at[idx, 'is_active'])
            for k, v in row.items():
                df.at[idx, k] = v
            display(HTML(f'<div style="color:#2980b9">✏️ {idx+1}번 행 수정 완료 (저장 전)</div>'))
        else:
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            display(HTML('<div style="color:#27ae60">✅ 신규 자산 추가 완료 (저장 전)</div>'))
        clear_form()
        render_table()

def on_del(b):
    """삭제 버튼 클릭 → 확인창 표시"""
    with msg_out:
        clear_output()
        idx = w_row_no.value - 1
        if 0 <= idx < len(df):
            name = df.at[idx, 'asset_name']
            confirm_box.children = [
                widgets.HTML(
                    f'<div style="background:#fdf2f8;border:1px solid #e74c3c;border-radius:6px;'
                    f'padding:10px 14px;margin:6px 0">'
                    f'🗑 <b>"{name}"</b>을(를) 정말 삭제하시겠습니까?<br>'
                    f'<span style="font-size:12px;color:#7f8c8d">삭제 후 저장하면 복구가 어렵습니다.</span></div>'
                ),
                widgets.HBox([btn_del_confirm, btn_del_cancel])
            ]
        else:
            display(HTML('<div style="color:#e74c3c">❌ 유효한 행 번호를 입력해 주세요.</div>'))

def on_del_confirm(b):
    """삭제 최종 확인"""
    global df
    confirm_box.children = []
    with msg_out:
        clear_output()
        idx = w_row_no.value - 1
        if 0 <= idx < len(df):
            name = df.at[idx, 'asset_name']
            df = df.drop(index=idx).reset_index(drop=True)
            clear_form()
            render_table()
            display(HTML(f'<div style="color:#e74c3c">🗑 "{name}" 삭제 완료 (저장 전)</div>'))

def on_del_cancel(b):
    """삭제 취소"""
    confirm_box.children = []
    with msg_out:
        clear_output()
        display(HTML('<div style="color:#7f8c8d">삭제 취소됐습니다.</div>'))

def on_deactivate(b):
    global df
    confirm_box.children = []
    with msg_out:
        clear_output()
        idx = w_row_no.value - 1
        if 0 <= idx < len(df):
            name = df.at[idx, 'asset_name']
            df.at[idx, 'is_active'] = 0
            render_table()
            display(HTML(
                f'<div style="color:#e67e22">🚫 "{name}" 비활성화 완료 (저장 전)<br>'
                f'<span style="font-size:12px;color:#7f8c8d">모든 계산에서 제외됩니다.</span></div>'
            ))
        else:
            display(HTML('<div style="color:#e74c3c">❌ 유효한 행 번호를 입력해 주세요.</div>'))

def on_reactivate(b):
    global df
    confirm_box.children = []
    with msg_out:
        clear_output()
        idx = w_row_no.value - 1
        if 0 <= idx < len(df):
            name = df.at[idx, 'asset_name']
            df.at[idx, 'is_active'] = 1
            render_table()
            display(HTML(f'<div style="color:#27ae60">✅ "{name}" 다시 활성화 완료 (저장 전)</div>'))
        else:
            display(HTML('<div style="color:#e74c3c">❌ 유효한 행 번호를 입력해 주세요.</div>'))

def on_save(b):
    confirm_box.children = []
    with msg_out:
        clear_output()
        import shutil
        from datetime import datetime
        bk = Path('backup'); bk.mkdir(exist_ok=True)
        if CSV_PATH.exists():
            stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            shutil.copy(CSV_PATH, bk / f'assets_{stamp}.csv')
        df.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
        a = int((df['is_active']==1).sum())
        n = int((df['is_active']==0).sum())
        display(HTML(
            f'<div style="color:#27ae60;font-weight:bold">'
            f'💾 assets.csv 저장 완료! 활성 {a}개 / 비활성 {n}개</div>'
            f'<div style="color:#7f8c8d;font-size:12px">기존 파일은 backup/ 폴더에 백업했습니다.</div>'
        ))

def on_clr(b):
    confirm_box.children = []
    with msg_out: clear_output()
    clear_form()

btn_add.on_click(on_add)
btn_del.on_click(on_del)
btn_del_confirm.on_click(on_del_confirm)
btn_del_cancel.on_click(on_del_cancel)
btn_deact.on_click(on_deactivate)
btn_react.on_click(on_reactivate)
btn_save.on_click(on_save)
btn_clr.on_click(on_clr)

display(HTML('<br><b style="font-size:13px;color:#555">자산 관리:</b>'))
display(widgets.HBox([btn_add, btn_del, btn_clr]))
display(confirm_box)
display(HTML('<b style="font-size:13px;color:#555;margin-top:6px;display:block">활성 상태 변경:</b>'))
display(widgets.HBox([btn_deact, btn_react, btn_save]))
display(msg_out)


Output()